In [13]:
#!pip install PyMuPDF
#!pip install sentence-transformers

In [1]:
import fitz  # PyMuPDF
from sentence_transformers import SentenceTransformer
import numpy as np

c:\Users\Yuan\miniconda3\envs\wh\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
import torch
import torch.distributed as dist

# Evita o erro em ambientes sem suporte a distributed (Windows, notebooks, etc.)
if not hasattr(dist, 'is_initialized'):
    dist.is_initialized = lambda: False

# RAG

In [3]:
import fitz  # PyMuPDF
from sentence_transformers import SentenceTransformer
import numpy as np

In [4]:
import torch
import torch.distributed as dist

# Evita o erro em ambientes sem suporte a distributed (Windows, notebooks, etc.)
if not hasattr(dist, 'is_initialized'):
    dist.is_initialized = lambda: False

In [5]:
# Passo 0: Instalar as bibliotecas necessárias
# Você precisa executar isso no seu terminal antes de rodar o script.
# pip install sentence-transformers pymupdf

import fitz  # PyMuPDF
from sentence_transformers import SentenceTransformer
import numpy as np

# --- PASSO 1: CARREGAR E EXTRAIR TEXTO DO PDF ---
def extrair_texto_do_pdf(caminho_pdf):
    """Abre um PDF e extrai o texto de todas as páginas."""
    doc = fitz.open(caminho_pdf)
    texto_completo = ""
    for pagina in doc:
        texto_completo += pagina.get_text()
    doc.close()
    return texto_completo

# --- PASSO 2: DIVIDIR O TEXTO EM PEDAÇOS (CHUNKS) ---
def dividir_em_chunks(texto, tamanho_chunk=500, sobreposicao=50):
    """Divide o texto em pedaços menores (chunks) com alguma sobreposição."""
    chunks = []
    inicio = 0
    while inicio < len(texto):
        fim = inicio + tamanho_chunk
        chunks.append(texto[inicio:fim])
        inicio += tamanho_chunk - sobreposicao
    return chunks

# --- PASSO 3: CRIAR OS EMBEDDINGS (VETORIZAÇÃO) ---
def criar_embeddings(chunks, nome_modelo):
    """
    Carrega um modelo de embedding e o utiliza para converter
    os chunks de texto em vetores numéricos.
    """
    # Carrega o modelo pré-treinado. O download será feito automaticamente na primeira vez.
    model = SentenceTransformer(nome_modelo)
    
    print(f"Criando embeddings para {len(chunks)} chunks de texto...")
    # A função encode pode processar uma lista de textos de uma só vez.
    embeddings = model.encode(chunks, show_progress_bar=True)
    print("Embeddings criados com sucesso!")
    
    return embeddings

# --- EXECUÇÃO PRINCIPAL ---
if __name__ == "__main__":
    # Configure aqui o caminho para o seu arquivo e o modelo desejado
    CAMINHO_DO_PDF = r"C:\Users\Yuan\Desktop\git_hub\ai_engineering\mcp\BEESDATA-Runbook - Delta Share - ITOPS-23784-100226-111522.pdf"  # Substitua pelo nome do seu arquivo PDF
    MODELO_EMBEDDING = './bge-small-en-v1.5' # Modelo recomendado: bom equilíbrio de performance e qualidade. [2, 4]

    # 1. Extrair texto
    texto_pdf = extrair_texto_do_pdf(CAMINHO_DO_PDF)
    if not texto_pdf.strip():
        print("AVISO: Nenhum texto foi extraído do PDF. Verifique se o arquivo não é uma imagem.")
    else:
        # 2. Dividir em chunks
        chunks_de_texto = dividir_em_chunks(texto_pdf)
        print(f"O texto foi dividido em {len(chunks_de_texto)} chunks.")

        # 3. Criar embeddings
        vetores_embedding = criar_embeddings(chunks_de_texto, MODELO_EMBEDDING)

        # Exibe informações sobre os embeddings gerados
        print(f"\nFormato do array de embeddings: {vetores_embedding.shape}")
        print(f"Cada chunk foi transformado em um vetor de {vetores_embedding.shape[1]} dimensões.")

        # O que fazer a seguir?
        # Agora você pode salvar `chunks_de_texto` e `vetores_embedding`
        # em um banco de dados vetorial (como ChromaDB, FAISS) para
        # realizar a busca semântica na sua aplicação RAG.
        # Por exemplo, para salvar em arquivos numpy:
        # np.save("embeddings.npy", vetores_embedding)
        # with open("chunks.txt", "w", encoding="utf-8") as f:
        #     for chunk in chunks_de_texto:
        #         f.write(chunk + "\n---\n")

O texto foi dividido em 5 chunks.
Criando embeddings para 5 chunks de texto...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]c:\Users\Yuan\miniconda3\envs\wh\Lib\site-packages\transformers\models\bert\modeling_bert.py:413: UserWarning: Mem Efficient attention on Current AMD GPU is still experimental. Enable it with TORCH_ROCM_AOTRITON_ENABLE_EXPERIMENTAL=1. (Triggered internally at C:/b/pytorch/aten/src/ATen/native/transformers/hip/sdp_utils.cpp:361.)
  attn_output = torch.nn.functional.scaled_dot_product_attention(
Batches: 100%|██████████| 1/1 [00:01<00:00,  1.26s/it]

Embeddings criados com sucesso!

Formato do array de embeddings: (5, 384)
Cada chunk foi transformado em um vetor de 384 dimensões.


In [6]:
import numpy as np
import pickle

np.save("embeddings.npy", vetores_embedding)

with open("chunks.pkl", "wb") as f:
    pickle.dump(chunks_de_texto, f)

print("Embeddings e chunks salvos.")

Embeddings e chunks salvos.


In [8]:
# carregar
import faiss
import numpy as np
import pickle

# Carrega os vetores numéricos (embeddings) já gerados.
embeddings = np.load("embeddings.npy")
with open("chunks.pkl", "rb") as f:
    chunks = pickle.load(f)

dim_ = embeddings.shape[0]
print(dim_)

dim = embeddings.shape[1]
print(dim)
# Descobre a dimensão dos vetores (ex: 384, 768, 1536…).

index = faiss.IndexFlatL2(dim)
index.add(embeddings)

#           query
#            ●
#           / \
#          /   \
#         ●     ●
#      chunk1 chunk2

print("Índice FAISS criado.")


index = faiss.IndexFlatL2(dim)
index.add(embeddings)

5
384
Índice FAISS criado.


In [9]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer('./bge-small-en-v1.5')

def buscar_contexto(pergunta, k=3):
    emb_pergunta = model.encode([pergunta])

    distancias, indices = index.search(emb_pergunta, k)

    resultados = []
    for idx in indices[0]:
        resultados.append(chunks[idx])

    return resultados

In [10]:
def montar_prompt(pergunta, contextos):
    contexto_texto = "\n\n".join(contextos)

    return f"""
<|system|>
Responda apenas com base no contexto abaixo.
Se a resposta não estiver no contexto, diga que não encontrou.
Responda em português.
</s>

<|user|>
CONTEXTO:
{contexto_texto}

PERGUNTA:
{pergunta}
</s>

<|assistant|>
"""

In [11]:
from gpt4all import GPT4All

llm = GPT4All(
    r"./tinyllama-1.1b-chat-v1.0.Q4_K_M.gguf",
    device="gpu"   # ou cpu
)

In [12]:
def responder_rag(pergunta):
    # busca no FAISS
    contextos = buscar_contexto(pergunta, k=4)

    # monta prompt
    prompt = montar_prompt(pergunta, contextos)

    # chama o LLM local
    with llm.chat_session():
        resposta = llm.generate(
            prompt,
            max_tokens=800,
            temp=0.2
        )

    return resposta


In [13]:
#pergunta = input("\nPergunta: ")
pergunta = "What is the Data Ops L2 process?"
resposta = responder_rag(pergunta)
print("\nResposta:\n", resposta)

c:\Users\Yuan\miniconda3\envs\wh\Lib\site-packages\transformers\models\bert\modeling_bert.py:413: UserWarning: Flash Efficient attention on Current AMD GPU is still experimental. Enable it with TORCH_ROCM_AOTRITON_ENABLE_EXPERIMENTAL=1. (Triggered internally at C:/b/pytorch/aten/src/ATen/native/transformers/hip/sdp_utils.cpp:310.)
  attn_output = torch.nn.functional.scaled_dot_product_attention(



Resposta:
 The Data Operations Level (L2) process refers to a specific set of procedures and guidelines that are designed to ensure the quality, reliability, and consistency of data produced by the organization. The L2 process is typically established for high-priority data sets or systems that require specialized expertise and resources to manage effectively. It involves defining clear roles and responsibilities, establishing metrics and performance indicators, setting up a governance structure, and implementing processes and procedures to ensure effective management of the data. The L2 process is typically overseen by a Data Operations Manager (L1) who ensures that all aspects of the data are managed effectively and efficiently.
